# Lab Stage 3: The Cloud-to-Edge Brain-Machine Interface

## Course architecture

Input design ➡️ On-device input processor ➡️ On-device output processor ➡️ Resource usage logger ➡️ Two-way control from the cloud

**Platform:** MicroPython-style runtime (simulated here in Python)
**Motivating domain:** real-time wearable and embedded medical devices
**Signal:** a simulated temperature sensor, 1000 readings per second

## Instructions for the participant
You have automated a pipeline (Lab 1) and scaled it to national research computing (Lab 2). Now you move it onto a small edge device: the [BBC micro:bit V2](https://microbit.org).

A micro:bit V2 has only **128 KB of working memory (RAM)** and no `numpy` or `scipy`. Processing large arrays on the chip would exhaust its memory immediately.

Use the **AI Prompt Challenges** to build
1. a simulated sensor input stream,
2. an on-device input processor that never runs out of memory,
3. an on-device output processor that drives the 5×5 LED display,
4. a resource usage logger, and
5. two-way control: a command from the cloud that changes the device's alert threshold.

You're welcome to try the official [micro:bit simulator](https://makecode.microbit.org/#editor), but here we simulate the device with the Python class below and fill in its methods step by step.

### ✅ Check the AI's code (every time)
Before you run any AI-generated code, ask three questions:
1. **Does it do what I asked?** Read it line by line against the prompt.
2. **Does it do anything I didn't ask for?** Downloads, file deletions, network calls, or changes to data it shouldn't touch.
3. **Can I explain each step in plain words?** If not, ask the AI to explain it before you run it.

On this device, also ask: **does it stay within the device's limits, and what happens when it gets bad input?**

The `MicroBit` class below simulates the device. Its methods are empty for now; you fill them in, module by module, by editing this cell and running it again.

In [ ]:

import numpy as np
import time
from datetime import datetime
from IPython.display import clear_output

class MicroBit:

    def __init__(self):
        self.MAX_RAM = 128 * 1024       # bytes
        self.MAX_STORAGE = 512 * 1024   # bytes
        self.BYTES_PER_VALUE = 8        # each reading is stored as an 8-byte float

        self.buffer = []
        self.storage = []
        self.threshold = 25.0           # alert threshold in deg C; the cloud can change it (Module 5)

    def ram_used(self):
        return len(self.buffer) * self.BYTES_PER_VALUE

    def _is_ram_available(self, extra_bytes=8):
        """True if `extra_bytes` more can be stored without going over MAX_RAM."""
        return self.ram_used() + extra_bytes <= self.MAX_RAM

    def output_to_screen(self, positions):
        clear_output(wait=True)
        grid = np.full((5, 5), '.', dtype=str)
        for r, c in positions:
            grid[r, c] = '*'
        for row in grid:
            print(' '.join(row))

    def read_from_temperature_sensor(self, input_stream):
        # TODO (Module 2): implement using AI prompting
        pass

    def process_temperature_data(self):
        # TODO (Module 3): implement using AI prompting
        pass

    def to_file(self, file):
        # TODO (Module 4): implement using AI prompting
        pass

    def receive_command(self, command):
        # TODO (Module 5): implement using AI prompting
        pass


***
## Module 1: Input design

On a real device, sensor readings arrive one at a time. Here we generate a whole signal in advance and feed it to the device one value at a time. This part runs on your laptop, standing in for the outside world, so it may use `numpy`.

**AI Prompt Challenge**

Copy this text block into your AI assistant:

> Write a python function generate_temperature_signal() that takes the input sampling rate in Hz, the length of the signal in ms, and a random seed (default = 123) and generates a 1d numpy array with temperature readings in celsius. Each temperature value should be randomly sampled from a 20-to-30-degrees interval. Each temperature reading is a sum of the actual temperature value and device-specific noise (normal with small SD). Use np.random.RandomState as RNG.

In [ ]:
# TODO: Paste your AI-generated code here, then run this cell.

Now generate a **5-second signal sampled at 1000 Hz** (5,000 readings) with seed 123, and plot it.

In [ ]:
signal = generate_temperature_signal(sampling_rate_hz=1000, length_ms=5000, random_seed=123)
print(len(signal), "readings")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 1, figsize=(6, 2))
ax.plot(signal, linewidth=0.5)
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Temperature (deg C)")
plt.show()

***
## Module 2: On-device input processor

**AI Prompt Challenge**

Copy the `MicroBit` class and this text block into your AI assistant:
> Write the method read_from_temperature_sensor, assuming input_stream is a 1d numpy array. It must read the array one element at a time using plain Python loops (the device has no numpy). Before storing each value in self.buffer, use self._is_ram_available(self.BYTES_PER_VALUE) to check that it still fits in RAM. If it does not fit, print "[ERROR]: Out of RAM!" and return 1. If the whole stream is read, print f"[SUCCESS] Used RAM: {self.ram_used()} bytes." and return 0.

Update the class above, run it again, then run the cell below.

In [ ]:
microbit = MicroBit()
microbit.read_from_temperature_sensor(signal)

***
## Module 3: On-device output processor

The micro:bit has a 5×5 grid of LEDs. Position `(0, 0)` is the top-left corner and `(2, 2)` the centre, often used as an alert light.

**AI Prompt Challenge**

Copy the `MicroBit` class and this text block into your AI assistant:
> Write the method process_temperature_data. It should read self.buffer one value at a time. If the value is above self.threshold (a class attribute that defaults to 25), light LED position (2, 2) to indicate an alert by calling self.output_to_screen([(2, 2)]); otherwise light position (0, 0) to indicate normal status.

Update the class above, run it again, then run the cell below. The display shows the device's current state: the last reading processed.

In [ ]:
microbit.process_temperature_data()

***
## Module 4: Resource usage logger

**AI Prompt Challenge**

Copy the `MicroBit` class and this text block into your AI assistant:
> Write the method to_file, which takes a file name and appends the current time (datetime.now().time()) and the current RAM used. If the file does not exist, the method should create it.

Update the class above, run it again, then run the cell below.

In [ ]:
microbit.to_file("out.txt")
print(open("out.txt").read())

***
## Module 5: Two-way control from the cloud

A real wearable doesn't only send data up; the cloud can send settings back down. Here the cloud looks at the readings the device has collected and sends it a new alert threshold, so that only **unusually high** readings (more than one standard deviation above the average) trigger the alert.

**AI Prompt Challenge**

Copy the `MicroBit` class and this text block into your AI assistant:
> Write a method receive_command(self, command) that accepts a dictionary such as {"type": "set_threshold", "value": 26.5}. If the type is "set_threshold" and the value is a number, set self.threshold to that value, print the new threshold and return 0. Otherwise print "[ERROR]: Unknown command", leave the threshold unchanged and return 1. Then, outside the class, write a cloud-side function cloud_set_threshold(device) that computes the mean plus one standard deviation of device.buffer with numpy and sends it to the device as a set_threshold command.

Put `receive_command` in the class above and run it again (then re-run Modules 2–3 so the device has data), and put `cloud_set_threshold` in the cell below.

In [ ]:
# TODO: Paste your AI-generated code here, then run this cell.

Run this cell as given. It counts the alerts before and after the cloud's command, then checks that bad commands are refused.

In [ ]:
alerts_before = sum(v > microbit.threshold for v in microbit.buffer)
cloud_set_threshold(microbit)
alerts_after = sum(v > microbit.threshold for v in microbit.buffer)
print(f"Alert readings: {alerts_before} before, {alerts_after} after "
      f"(out of {len(microbit.buffer)}).")

# Check the AI's code: these must be refused, and the threshold must not change
current = microbit.threshold
microbit.receive_command({"type": "reboot"})
microbit.receive_command({"type": "set_threshold", "value": "hot"})
assert microbit.threshold == current, "A bad command changed the threshold!"
print("OK: bad commands were refused.")

***
## Embedded Playground

**Task 1.** Generate input that makes the micro:bit run out of RAM.

**Task 2.** Modify `read_from_temperature_sensor()` to report used RAM while it reads, not only at the end.

**Task 3.** Modify `process_temperature_data()` to display the first digit of the temperature.

**Task 4.** Prompt AI to make it possible to store sensor data in the micro:bit's internal storage (`self.storage`, limited to `self.MAX_STORAGE`).